# Adam Optimization

Implement Adam from scratch with bias-corrected moment estimates, validate against `torch.optim.Adam`, and visualize convergence compared to SGD, SGD-Momentum, and RMSProp.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()` — never hardcoded. On Apple Silicon this runs on mps.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


## The problem

We optimize a small linear regression: \(y = Xw^* + \epsilon\) with \(D = 10\) features. This is convex and has a unique minimum, making it a clean sandbox for optimizer comparison. We use mini-batch GD (full-batch here) to keep the comparison fair.

In [2]:
torch.manual_seed(0)
N, D = 128, 10

# True weights and data on device
true_w = torch.randn(D, device=device)
X = torch.randn(N, D, device=device)
y = X @ true_w + 0.1 * torch.randn(N, device=device)


def mse(w: torch.Tensor) -> torch.Tensor:
    return ((X @ w - y) ** 2).mean()


print(f"Data: X{tuple(X.shape)}, y{tuple(y.shape)}, true_w{tuple(true_w.shape)}")
print(f"Optimal loss (closed-form approx): {mse(true_w).item():.6f}")


Data: X(128, 10), y(128,), true_w(10,)


Optimal loss (closed-form approx): 0.007905


## Adam from scratch

Adam (Kingma & Ba, 2015) maintains two exponential moving averages of the gradient:

- **First moment** \(m_t\): momentum (smoothed gradient direction)
- **Second moment** \(v_t\): adaptive scale (smoothed squared gradient)

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

**Bias correction.** Because \(m_0 = v_0 = 0\), early estimates are biased toward zero — the first update would use a moment of \((1-\beta_1)g_1\) instead of \(g_1\). Dividing by \(1 - \beta_1^t\) (respectively \(1 - \beta_2^t\)) corrects this: as \(t \to \infty\), the correction approaches 1 and has no effect.

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

Parameter update:

$$\theta_t = \theta_{t-1} - \alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

In [3]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class AdamState:
    """Immutable Adam optimizer state."""

    w: torch.Tensor
    m: torch.Tensor   # first moment (momentum)
    v: torch.Tensor   # second moment (adaptive scale)
    t: int            # step counter


def adam_init(w0: torch.Tensor) -> AdamState:
    """Initialize Adam state from a starting parameter tensor."""
    zeros = torch.zeros_like(w0)
    return AdamState(w=w0.clone(), m=zeros, v=zeros, t=0)


def adam_step(
    state: AdamState,
    grad: torch.Tensor,
    lr: float = 1e-3,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
) -> AdamState:
    """Return a new AdamState after one update step (immutable).

    The bias correction divides by (1 - beta^t) so that early steps,
    where the exponential moving average is biased toward zero, are
    corrected to their true expected values.
    """
    t = state.t + 1
    m_new = beta1 * state.m + (1 - beta1) * grad          # first moment EMA
    v_new = beta2 * state.v + (1 - beta2) * grad ** 2     # second moment EMA
    m_hat = m_new / (1 - beta1 ** t)                       # bias-corrected first
    v_hat = v_new / (1 - beta2 ** t)                       # bias-corrected second
    w_new = state.w - lr * m_hat / (v_hat.sqrt() + eps)
    return AdamState(w=w_new, m=m_new, v=v_new, t=t)


print("AdamState and adam_step defined.")


AdamState and adam_step defined.


In [4]:
def run_scratch_adam(
    w0: torch.Tensor,
    loss_fn,
    steps: int,
    lr: float = 1e-3,
) -> tuple[list[float], torch.Tensor]:
    """Run from-scratch Adam for `steps` iterations. Returns (loss history, final w)."""
    state = adam_init(w0)
    hist: list[float] = []
    for _ in range(steps):
        w = state.w.detach().requires_grad_(True)
        loss = loss_fn(w)
        loss.backward()
        state = adam_step(state, w.grad, lr=lr)
        hist.append(loss.item())
    return hist, state.w.detach()


STEPS = 500
LR = 1e-2
w0 = torch.zeros(D, device=device)

hist_scratch, w_scratch = run_scratch_adam(w0, mse, STEPS, lr=LR)
print(f"Scratch Adam final loss: {hist_scratch[-1]:.6f}")
print(f"Scratch Adam final w[:4]: {w_scratch[:4].tolist()}")


Scratch Adam final loss: 0.007247
Scratch Adam final w[:4]: [-0.3312992453575134, -1.4362962245941162, 0.7698742151260376, -1.1678463220596313]


## Validation against `torch.optim.Adam`

We run `torch.optim.Adam` on the same problem with the same hyperparameters and assert that the final weights agree within a tight tolerance.

In [5]:
w_torch = torch.zeros(D, device=device, requires_grad=True)
opt_torch = torch.optim.Adam([w_torch], lr=LR)
hist_torch: list[float] = []

for _ in range(STEPS):
    opt_torch.zero_grad()
    loss_val = mse(w_torch)
    loss_val.backward()
    opt_torch.step()
    hist_torch.append(loss_val.item())

w_torch_final = w_torch.detach()
print(f"Torch  Adam final loss: {hist_torch[-1]:.6f}")
print(f"Torch  Adam final w[:4]: {w_torch_final[:4].tolist()}")

max_diff = (w_scratch - w_torch_final).abs().max().item()
print(f"Max absolute diff in final weights: {max_diff:.6f}")
assert torch.allclose(w_scratch, w_torch_final, atol=1e-2), (
    f"From-scratch Adam diverges from torch.optim.Adam: max diff = {max_diff}"
)
print("From-scratch Adam matches torch.optim.Adam within atol=1e-2 ✓")


Torch  Adam final loss: 0.007247
Torch  Adam final w[:4]: [-0.3312992453575134, -1.4362962245941162, 0.7698742151260376, -1.1678463220596313]
Max absolute diff in final weights: 0.000000
From-scratch Adam matches torch.optim.Adam within atol=1e-2 ✓


## Effect of bias correction

We compare Adam with and without bias correction on the same problem. Without correction, early moment estimates are deflated toward zero, causing a slow start and unstable early updates.

In [6]:
def adam_step_no_correction(
    state: AdamState,
    grad: torch.Tensor,
    lr: float = 1e-3,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
) -> AdamState:
    """Adam step WITHOUT bias correction (for comparison only — do not use)."""
    t = state.t + 1
    m_new = beta1 * state.m + (1 - beta1) * grad
    v_new = beta2 * state.v + (1 - beta2) * grad ** 2
    # Skips bias correction: uses m, v directly
    w_new = state.w - lr * m_new / (v_new.sqrt() + eps)
    return AdamState(w=w_new, m=m_new, v=v_new, t=t)


def run_adam_variant(step_fn, w0, loss_fn, steps, lr=1e-2):
    state = adam_init(w0)
    hist = []
    for _ in range(steps):
        w = state.w.detach().requires_grad_(True)
        loss = loss_fn(w)
        loss.backward()
        state = step_fn(state, w.grad, lr=lr)
        hist.append(loss.item())
    return hist


hist_corrected = run_adam_variant(adam_step, w0, mse, STEPS, LR)
hist_uncorrected = run_adam_variant(adam_step_no_correction, w0, mse, STEPS, LR)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist_corrected[:50], label="Adam (with bias correction)")
ax.plot(hist_uncorrected[:50], label="Adam (no bias correction)", linestyle="--")
ax.set_xlabel("step")
ax.set_ylabel("MSE loss")
ax.set_title("Early steps: bias correction matters")
ax.legend()
plt.tight_layout()
plt.show()
print("Early-step loss with correction:   ", hist_corrected[:5])
print("Early-step loss without correction:", hist_uncorrected[:5])


Early-step loss with correction:    [8.193367004394531, 8.020368576049805, 7.849780082702637, 7.681641578674316, 7.515990257263184]
Early-step loss without correction: [8.193367004394531, 7.65431547164917, 6.967781066894531, 6.223383903503418, 5.473425388336182]


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_17212/1968149389.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Adam vs SGD / Momentum / RMSProp

We implement all four optimizers from scratch and compare convergence on the same regression problem. Each optimizer is tuned to a reasonable learning rate; the comparison is illustrative, not a rigorous benchmark.

> **Note:** The  below uses the EMA-scaled convention `v = β·v + (1−β)·grad`; `torch.optim.SGD(momentum=β)` instead uses the heavy-ball form `v = β·v + grad` (no `(1−β)` scaling), so its effective step size differs by a factor of `(1−β)`.

In [7]:
@dataclass(frozen=True)
class SGDState:
    w: torch.Tensor


@dataclass(frozen=True)
class MomState:
    w: torch.Tensor
    v: torch.Tensor  # velocity


@dataclass(frozen=True)
class RMSState:
    w: torch.Tensor
    s: torch.Tensor  # squared-gradient EMA


def sgd_step(state: SGDState, grad, lr=0.01):
    return SGDState(w=state.w - lr * grad)


def momentum_step(state: MomState, grad, lr=0.01, beta=0.9):
    # EMA-scaled convention: v = β·v + (1−β)·grad  (differs from torch.optim.SGD heavy-ball
    # form v = β·v + grad by a factor of (1−β) in the effective step size)
    v = beta * state.v + (1 - beta) * grad
    return MomState(w=state.w - lr * v, v=v)


def rms_step(state: RMSState, grad, lr=0.01, beta=0.99, eps=1e-8):
    s = beta * state.s + (1 - beta) * grad ** 2
    return RMSState(w=state.w - lr * grad / (s.sqrt() + eps), s=s)


def run_optimizer(init_state, step_fn, loss_fn, steps):
    state = init_state
    hist = []
    for _ in range(steps):
        w = state.w.detach().requires_grad_(True)
        loss = loss_fn(w)
        loss.backward()
        state = step_fn(state, w.grad)
        hist.append(loss.item())
    return hist


zeros = torch.zeros(D, device=device)
configs = {
    "SGD": (SGDState(w=zeros.clone()), lambda s, g: sgd_step(s, g, lr=0.05)),
    "Momentum": (MomState(w=zeros.clone(), v=zeros.clone()), lambda s, g: momentum_step(s, g, lr=0.02)),
    "RMSProp": (RMSState(w=zeros.clone(), s=zeros.clone()), lambda s, g: rms_step(s, g, lr=0.01)),
    "Adam": (adam_init(zeros.clone()), lambda s, g: adam_step(s, g, lr=LR)),
}

fig, ax = plt.subplots(figsize=(9, 5))
for name, (init, step_fn) in configs.items():
    hist = run_optimizer(init, step_fn, mse, STEPS)
    ax.plot(hist, label=name)
ax.set_xlabel("step")
ax.set_ylabel("MSE loss")
ax.set_yscale("log")
ax.set_title("Optimizer convergence (log scale)")
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_17212/3977867589.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## The idiomatic PyTorch way

Use `torch.optim.Adam` or `torch.optim.AdamW`. AdamW is preferred for modern deep learning: it applies weight decay *before* the gradient scaling step (decoupled), whereas Adam+L2 applies weight decay *through* the second-moment scaling (coupled), which reduces the effective regularization strength for parameters with large gradient variance.

In [8]:
# torch.optim.Adam
w_a = torch.zeros(D, device=device, requires_grad=True)
adam = torch.optim.Adam([w_a], lr=LR)

# torch.optim.AdamW (preferred for regularized training)
w_aw = torch.zeros(D, device=device, requires_grad=True)
adamw = torch.optim.AdamW([w_aw], lr=LR, weight_decay=1e-3)

for _ in range(STEPS):
    for opt, w in [(adam, w_a), (adamw, w_aw)]:
        opt.zero_grad()
        mse(w).backward()
        opt.step()

print(f"torch.optim.Adam  final loss: {mse(w_a).item():.6f}")
print(f"torch.optim.AdamW final loss: {mse(w_aw).item():.6f}")
# Both should converge; Adam to near 0, AdamW slightly higher due to regularization
assert mse(w_a).item() < 0.05
assert mse(w_aw).item() < 0.1
print("Both Adam and AdamW converge ✓")


torch.optim.Adam  final loss: 0.007247
torch.optim.AdamW final loss: 0.007248
Both Adam and AdamW converge ✓


## Takeaways

- Adam maintains a **first moment** \(m_t\) (momentum: exponential moving average of gradients) and a **second moment** \(v_t\) (scale: EMA of squared gradients).
- **Bias correction** divides by \(1 - \beta^t\) because the moments are initialized to zero. Without correction, early estimates are biased toward zero, making the first updates uselessly small. This is not 'symbolic differentiation' — it is a statistical correction for the cold-start of a running average.
- The effective step size per parameter is \(\alpha / (\sqrt{\hat{v}_t} + \epsilon)\): parameters with large historical gradients receive smaller effective steps, providing per-coordinate adaptation.
- **AdamW decouples weight decay**: it applies \(w \leftarrow (1 - \alpha\lambda) w\) before the gradient step, so weight decay is not scaled by the second moment. This is the recommended default for modern deep learning.
- Default hyperparameters (\(\beta_1=0.9\), \(\beta_2=0.999\), \(\epsilon=10^{-8}\)) work well across a wide range of problems; the learning rate \(\alpha\) usually requires tuning and/or warmup.